In [5]:
import re
import requests
import pandas as pd
from urllib.parse import urljoin


BASE = "https://www.aspeedtech.com"
PAGE = "/financials_monthly/"


def _session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9,ko;q=0.8",
    })
    return s


def _find_monthly_js_url(html: str) -> str:
    # <script src='/public/js/monthly-dist.min.js'></script> 찾기
    m = re.search(r"""<script[^>]+src=['"]([^'"]*monthly-dist\.min\.js)['"]""", html, re.I)
    if not m:
        raise RuntimeError("monthly-dist.min.js를 HTML에서 찾지 못했습니다.")
    return urljoin(BASE, m.group(1))


def _extract_endpoints(js_text: str) -> list[str]:
    """
    JS 안에서 데이터 호출 URL 후보를 뽑습니다.
    (fetch / axios / $.ajax / xhr open 등)
    """
    cands = set()

    # fetch("..."), axios.get("..."), $.get("..."), open("GET","...")
    patterns = [
        r"""fetch\(\s*["']([^"']+)["']""",
        r"""axios\.(?:get|post)\(\s*["']([^"']+)["']""",
        r"""\$\.(?:get|post)\(\s*["']([^"']+)["']""",
        r"""open\(\s*["']GET["']\s*,\s*["']([^"']+)["']""",
        r"""url\s*:\s*["']([^"']+)["']""",
    ]
    for pat in patterns:
        for m in re.finditer(pat, js_text, re.I):
            cands.add(m.group(1).strip())

    # 우선순위 정렬(월매출/financial/api 등 포함된 것 먼저)
    out = [x for x in cands if x and len(x) < 200]
    out.sort(key=lambda x: (0 if re.search(r"(monthly|revenue|financial|api|json)", x, re.I) else 1, len(x)))
    return out


def _normalize_payload_to_df(payload, year: int) -> pd.DataFrame:
    """
    payload 구조는 사이트 구현에 따라 다름.
    dict/list 형태를 최대한 유연하게 DataFrame으로 변환.
    """
    if isinstance(payload, list):
        df = pd.DataFrame(payload)
    elif isinstance(payload, dict):
        # dict 안에 list가 들어있는 흔한 케이스 탐색
        list_key = None
        for k, v in payload.items():
            if isinstance(v, list) and v:
                list_key = k
                break
        df = pd.DataFrame(payload[list_key]) if list_key else pd.DataFrame([payload])
    else:
        raise RuntimeError("JSON payload 형식을 이해할 수 없습니다.")

    df.insert(0, "year", year)
    return df


def fetch_aspeed_monthly_revenue_df(year: int = 2025) -> pd.DataFrame:
    s = _session()

    # 1) 페이지 HTML
    page_url = f"{BASE}{PAGE}?years={year}"
    html = s.get(page_url, timeout=20).text

    # 2) monthly JS 다운로드
    js_url = _find_monthly_js_url(html)
    js_text = s.get(js_url, timeout=20).text

    # 3) JS에서 endpoint 후보 추출
    endpoints = _extract_endpoints(js_text)
    if not endpoints:
        raise RuntimeError("JS에서 데이터 엔드포인트 후보를 찾지 못했습니다. (난독화/동적 생성 가능)")

    # 4) 후보 endpoint에 year 파라미터를 붙여 JSON 응답 시도
    year_keys = ["years", "year", "y", "yr"]
    last_err = None

    for ep in endpoints:
        # 상대/절대 URL 정규화
        if ep.startswith("//"):
            url_base = "https:" + ep
        elif ep.startswith("/"):
            url_base = urljoin(BASE, ep)
        elif ep.lower().startswith("http"):
            url_base = ep
        else:
            url_base = urljoin(f"{BASE}{PAGE}", ep)

        for k in year_keys:
            try_url = url_base + ("&" if "?" in url_base else "?") + f"{k}={year}"
            try:
                r = s.get(try_url, timeout=20)
                if r.status_code != 200:
                    continue
                txt = r.text.strip()
                if (r.headers.get("Content-Type", "").lower().find("json") >= 0) or txt.startswith(("{", "[")):
                    payload = r.json()
                    return _normalize_payload_to_df(payload, year)
            except Exception as e:
                last_err = e
                continue

    raise RuntimeError(f"JSON 엔드포인트 자동 탐색 실패. last_err={last_err}\n"
                       f"DevTools(Network)에서 XHR/Fetch로 호출되는 URL을 확인해 주시면 100% 고정 코드로 드릴 수 있습니다.")


In [6]:
import asyncio
df_aspeed = fetch_aspeed_monthly_revenue_df(2025)
df_aspeed

RuntimeError: JS에서 데이터 엔드포인트 후보를 찾지 못했습니다. (난독화/동적 생성 가능)

In [20]:
import re
import ast
import pandas as pd


def _to_dict(x):
    """
    monthlyRevenue 셀 값이
    - dict 이면 그대로
    - 문자열("{'m1': 123, ...}") 이면 dict로 파싱
    - 그 외/결측이면 {}
    """
    if isinstance(x, dict):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return {}
        # dict 문자열 파싱 (python literal 형태)
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, dict) else {}
        except Exception:
            return {}
    return {}


def _month_key_to_int(k):
    """
    'm1' -> 1, 'm01' -> 1, 'M12' -> 12 같은 케이스 대응
    """
    if k is None:
        return None
    m = re.match(r"^[mM]\s*0*([1-9]|1[0-2])$", str(k).strip())
    return int(m.group(1)) if m else None


def expand_monthly_revenue(df: pd.DataFrame,
                           year_col: str = "year",
                           revenue_col: str = "monthlyRevenue",
                           keep_cols=None,
                           make_date: bool = True) -> pd.DataFrame:
    """
    df[revenue_col] 안의 {'m1':..., 'm2':...} 를 연/월 단위로 펼쳐 long DF로 반환.
    - keep_cols: year와 monthlyRevenue 외에 함께 보존할 원본 컬럼 리스트(예: 회사명 등)
    - make_date=True면 month_end 날짜 컬럼(date) 추가
    """
    if keep_cols is None:
        keep_cols = []

    # 필요한 컬럼만 추림
    base_cols = [year_col] + keep_cols + [revenue_col]
    missing = [c for c in base_cols if c not in df.columns]
    if missing:
        raise KeyError(f"다음 컬럼이 df에 없습니다: {missing}")

    work = df[base_cols].copy()
    work[revenue_col] = work[revenue_col].map(_to_dict)

    records = []
    for _, row in work.iterrows():
        year = int(row[year_col])
        meta = {c: row[c] for c in keep_cols}
        rev_dict = row[revenue_col] or {}

        for k, v in rev_dict.items():
            m_int = _month_key_to_int(k)
            if m_int is None:
                continue

            # 값 정리: None/'None'/NaN -> NaN, 문자열 숫자 -> numeric
            if v is None or (isinstance(v, str) and v.strip().lower() == "none"):
                val = pd.NA
            else:
                val = v
            records.append({
                year_col: year,
                **meta,
                "month": m_int,
                "monthly_revenue": val
            })

    out = pd.DataFrame(records)

    if out.empty:
        # 데이터가 전혀 없을 때도 컬럼은 유지
        out = pd.DataFrame(columns=[year_col] + keep_cols + ["month", "monthly_revenue"])

    # 숫자 변환
    out["monthly_revenue"] = pd.to_numeric(out["monthly_revenue"], errors="coerce")

    # 정렬
    out = out.sort_values([year_col, "month"]).reset_index(drop=True)

    # date(월말) 컬럼 생성 옵션
    if make_date:
        out["date"] = pd.to_datetime(
            out[year_col].astype(str) + "-" + out["month"].astype(str).str.zfill(2) + "-01"
        ) + pd.offsets.MonthEnd(0)

    return out

def assign_reverse_year_dates(
    df: pd.DataFrame,
    month_col: str = "month",
    date_col_out: str = "date_new",
    start_year: int = 2026,
    end_year: int = 2011,
    month_end: bool = True,
    check_group_size: bool = True,
) -> pd.DataFrame:
    """
    규칙:
    - 같은 month 값 안에서 위에서부터 순서대로 start_year, start_year-1, ..., end_year 를 배정
    - month_end=True면 각 월의 월말 날짜로 date_new 생성, False면 YYYY-MM-01
    """
    out = df.copy()

    years = list(range(start_year, end_year - 1, -1))  # 2026..2011 (16개)
    expected_n = len(years)

    # month별로 "현재 정렬된 순서"를 유지하려면 groupby(sort=False)
    def _apply(group: pd.DataFrame) -> pd.DataFrame:
        n = len(group)
        if check_group_size and n != expected_n:
            raise ValueError(
                f"month={group.name} 그룹의 행 개수={n} (기대={expected_n}). "
                f"규칙 적용 불가: 데이터 정렬/중복/누락 여부 확인 필요."
            )

        # 위에서부터 순서대로 연도 할당
        assign_years = years[:n]
        g = group.copy()
        g["_assign_year"] = assign_years

        # 날짜 생성
        base = pd.to_datetime(
            g["_assign_year"].astype(str) + "-" + g[month_col].astype(int).astype(str).str.zfill(2) + "-01"
        )
        g[date_col_out] = base + (pd.offsets.MonthEnd(0) if month_end else pd.offsets.Day(0))

        return g.drop(columns=["_assign_year"])

    out = (
        out.groupby(month_col, sort=False, group_keys=False)
           .apply(_apply)
           .reset_index(drop=True)
    )

    df_sorted = (
    out.sort_values("date", ascending=True)
    .reset_index(drop=True))

    return df_sorted


In [17]:
# df_raw: 질문에 나온 원본 데이터프레임
df_monthly = expand_monthly_revenue(df_aspeed, year_col="year", revenue_col="monthlyRevenue")


In [4]:
df_fixed = assign_reverse_year_dates(df_monthly, month_col="month", date_col_out="date", month_end=True)
df_fixed[['monthly_revenue']].plot()

NameError: name 'assign_reverse_year_dates' is not defined